# STAT 386: data cleaning and string operations
These talk records are illustrative. The titles and word counts are invented. Work through the prediction prompts before running each cell. Open the slides alongside the notebook when you want to compare outputs.

**Goal:** preserve raw values, write repeatable rules, inspect results, and decide whether two processed batches can be combined.

## 1. Warmup: missingness
Suppose a word count is more likely to be missing for talks with a *long recorded duration*. The import bug does not use the missing word count itself. Let $R=1$ mean the count is missing, $Y$ be the count, and $X$ be the observed duration.

**Predict:** Which missingness category fits the stated mechanism? What could a plot of $R$ against duration suggest, and what can it not prove?

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("data")
reference = pd.read_csv(DATA / "talks_reference.csv")
new = pd.read_csv(DATA / "talks_new.csv")
print("Reference records")
print(reference.to_string(index=False))
print("\nNew records")
print(new.to_string(index=False))

## 2. Inspect before writing rules
**With a partner:** Which speaker names should refer to the same person? Which raw field should stay untouched so we can change our cleaning rule later?

In [ ]:
for label, batch in [("reference", reference), ("new", new)]:
    print(label, [repr(value) for value in batch["speaker"].tolist()])

### Predict the string method output
What do you expect for the two Russell M. Nelson rows and for the missing speaker? The string strip method removes spaces at the beginning and end. Does it change spaces in the middle?

In [ ]:
old_key = reference["speaker"].astype("string").str.strip().str.lower()
new_key = new["speaker"].astype("string").str.strip().str.lower()
print("Reference key:", repr(old_key.iloc[0]))
print("New key:      ", repr(new_key.iloc[0]))
print("Missing key:  ", repr(new_key.iloc[2]))
print("Keys match:", old_key.iloc[0] == new_key.iloc[0])

## 3. Write one repeatable function
The three reference counts are 1000, 2000, and 3000. Their mean is 2000 and their **population** standard deviation is 1000. We keep these values fixed when we process the new records.

First predict: will the function change the input DataFrame? What is the z-score for a new value of 3500?

In [ ]:
def clean_talks(raw: pd.DataFrame) -> pd.DataFrame:
    result = raw.copy()
    result["speaker_key"] = (
        result["speaker"].astype("string").str.strip().str.lower()
    )
    result["word_count"] = pd.to_numeric(result["word_count"], errors="raise")
    result["word_scaled"] = (result["word_count"] - 2000) / 1000
    return result

cleaned_reference = clean_talks(reference)
print(cleaned_reference[["talk_id", "speaker_key", "word_count", "word_scaled"]].to_string(index=False))
print("Input unchanged:", "speaker_key" not in reference.columns)

### Checks tied to our claims
What should be true of IDs and counts? We do **not** assert that every speaker is known, since the new data has an unknown speaker.

In [ ]:
assert cleaned_reference["talk_id"].is_unique
assert cleaned_reference["talk_id"].notna().all()
assert cleaned_reference["word_count"].ge(0).all()
assert "speaker_key" not in reference.columns
print("Reference checks passed.")

## 4. Think, pair, share: new records
**Before running:** Predict the speaker key and scaled count for T04. Will T04 match T01? What happens to T06's missing speaker? Suggest a better rule and a check that could reveal the problem.

In [ ]:
cleaned_new = clean_talks(new)
print(cleaned_new[["talk_id", "speaker_key", "word_scaled"]].to_string(index=False))
print("T01 and T04 keys match:", cleaned_reference.loc[0, "speaker_key"] == cleaned_new.loc[0, "speaker_key"])

One repair uses split and join methods. With no separator, split treats runs of whitespace as one boundary. **Predict** the result for the two strings and the missing speaker.

In [ ]:
for batch in [reference, new]:
    repaired = batch["speaker"].astype("string").str.split().str.join(" ").str.lower()
    print([repr(value) for value in repaired.tolist()])

## 5. What if the pipeline changes?
The separate functions in cleaning.py implement the two versions shown in the slides. They keep the original columns, add a processed key and scale, and record the version and reference identifier. We inspect them here, rather than implementing a versioning system during class.

In [ ]:
from cleaning import (
    clean_v1, clean_v2, check_cleaned, PIPELINE_V1, PIPELINE_V2,
    MEAN_WORDS, STD_WORDS, MIN_WORDS, MAX_WORDS,
)

old_v1 = clean_v1(reference)
new_v1 = clean_v1(new)
old_v2 = clean_v2(reference)
new_v2 = clean_v2(new)
for result, version in [(old_v1, PIPELINE_V1), (new_v1, PIPELINE_V1),
                        (old_v2, PIPELINE_V2), (new_v2, PIPELINE_V2)]:
    check_cleaned(result, version)
print("All processed batches passed the basic checks.")

**Predict before running:** Does T04 under v2 get a value between zero and one? The v2 minimum and maximum come from the *reference* records, not this new batch.

In [ ]:
print("Saved v1 mean/std:", MEAN_WORDS, STD_WORDS)
print("Saved v2 min/max:", MIN_WORDS, MAX_WORDS)
print(pd.concat([old_v1.iloc[[0]], new_v1.iloc[[0]], old_v2.iloc[[0]], new_v2.iloc[[0]]])[
    ["talk_id", "speaker_key", "word_count", "word_scaled", "pipeline_version", "scaling_reference"]
].to_string(index=False))

### Compatibility discussion
A teammate has old v1 records and new v2 records. For each field, decide whether the batches can be combined as they stand:

| Field | Decision and reason |
|---|---|
| Original speaker, talk_id, word_count | |
| speaker_key | |
| word_scaled | |

Suppose both batches were labeled v2, but each chose its own minimum and maximum. Would the label alone make their scaled values comparable? What would you record to settle that question?

In [ ]:
# One safe route: apply v2 to both raw batches with the SAME saved reference values.
compatible = pd.concat([clean_v2(reference), clean_v2(new)], ignore_index=True)
check_cleaned(compatible, PIPELINE_V2)
assert compatible["talk_id"].is_unique
assert compatible.loc[compatible["talk_id"].eq("T01"), "speaker_key"].iloc[0] == (
    compatible.loc[compatible["talk_id"].eq("T04"), "speaker_key"].iloc[0]
)
print(compatible[["talk_id", "speaker_key", "word_count", "word_scaled"]].to_string(index=False))

## Exit check
1. Why preserve raw speaker text as well as speaker_key?
2. Why do the code version **and** saved numeric reference matter?
3. If a new talk has 3500 words, why can its v2 value be 1.25?
4. Which string problems would need more flexible matching than today's methods? Next class: regular expressions.